# Unit 09 — 転移学習で画像分類を組み立てる

目安は **20〜25分**。前ユニットでは画像配列と古典特徴を扱いました。今回は、画像を渡す入口から予測を評価する出口までを、転移学習の部品に分けます。

前半はshapeを見やすい NumPy 模型で契約を分解し、後半はインストール済みの `torch` / `torchvision` で同じ処理を完全オフライン実行します。完成時には次を説明・実装できます。

- `Dataset` が1件、`DataLoader` が `(B, C, H, W)` のバッチを返す契約
- 汎用 `backbone` と分類 `head`、freeze / fine-tune の違い
- 学習対象パラメータだけを optimizer に渡す理由
- confusion matrix とクラス別指標、TTA の予測平均
- CPU/GPU、解像度、バッチ、TTA のコスト判断

> C# なら `Dataset` は `IReadOnlyList<Sample>`、`DataLoader` は列挙・シャッフル・まとめ上げを担う `IEnumerable<Batch>` に近い役割です。

In [ ]:
# STEP 1: Unit 09 — 転移学習で画像分類を組み立てるの処理を実行し、出力を照合する
from pathlib import Path
import copy
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

DATA_DIR = Path("../unit08-image-and-pattern-basics/data")
if not DATA_DIR.exists():
    DATA_DIR = Path("courses/kaggle-sprint/unit08-image-and-pattern-basics/data")
if not DATA_DIR.exists():
    raise FileNotFoundError("unit08 の data が見つかりません。unit09 またはリポジトリ直下から実行してください。")

train_df = pd.read_csv(DATA_DIR / "train.csv")
print(f"data={DATA_DIR.resolve()}")
print(f"rows={len(train_df)}, classes={train_df['label'].nunique()}, products={train_df['product_key'].nunique()}")

def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(
            _np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float)
        )))
    except (TypeError, ValueError):
        ok = actual == expected
    mark = "OK" if ok else "NG"
    if ok:
        print(f"[OK] {name}")
    else:
        detail = f"actual={actual!r} / expected={expected!r}"
        print(f"[NG] {name} — {detail}" + ("" if not hint else f" — ヒント: {hint}"))
    return ok

def safe_call(fn):
    try:
        return fn()
    except Exception:
        return None

def safe_get(mapping, key):
    return mapping.get(key) if isinstance(mapping, dict) else None

## 今日の流れ

各ブロックを **見る → 予測 → 変える → 書く → チェック** の順で進めます。

1. `Dataset` と transform 注入
2. `DataLoader` と HWC → CHW バッチ
3. backbone / head と freeze / fine-tune
4. TTA と計算・メモリコスト
5. confusion matrix と誤分類確認

実際の PyTorch コードへ移るときも境界は同じです。ここでは仕組みを小さく見える形にし、重みのダウンロードや GPU の有無を学習条件にしません。

## 1. `Dataset` は「1件を返す」

`torch.utils.data.Dataset` は `__len__` と `__getitem__` を実装する基底クラスです。`__getitem__(i)` は通常、画像とラベルを1件だけ返します。画像処理は constructor で `transform` を受け取り、取得時に呼ぶと差し替え可能です。

C# の constructor injection と同じです。画像処理を class 内に固定せず、`Func<Image, Tensor>` 相当を注入すれば、訓練時と検証時で処理を替えられます。検証 transform は決定的にし、訓練だけにランダム拡張を使います。

In [ ]:
# STEP 2: 1. `Dataset` は「1件を返す」の処理を実行し、出力を照合する
def to_chw_float(image_hwc):
    # HWC uint8 を CHW float32 [0, 1] にする。ToTensor 相当。
    return np.transpose(image_hwc.astype(np.float32) / 255.0, (2, 0, 1))

class ReferenceArrayDataset:
    def __init__(self, images, labels, transform=None):
        self.images, self.labels, self.transform = images, labels, transform
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        image = self.images[index].copy()
        if self.transform is not None:
            image = self.transform(image)
        return {"image": image, "label": int(self.labels[index])}

tiny_images = np.arange(2 * 2 * 2 * 3, dtype=np.uint8).reshape(2, 2, 2, 3)
tiny_labels = np.array([3, 1])
reference_dataset = ReferenceArrayDataset(tiny_images, tiny_labels, transform=to_chw_float)
reference_dataset[0]

In [ ]:
# STEP 3: 1. `Dataset` は「1件を返す」の処理を実行し、出力を照合する
sample = reference_dataset[0]
print("len:", len(reference_dataset))
print("image:", sample["image"].shape, sample["image"].dtype)
print("label:", sample["label"])
print("first pixel RGB -> CHW:", tiny_images[0, 0, 0].tolist(), sample["image"][:, 0, 0].round(4).tolist())

### 予測

`transform=None` にすると、返る画像 shape はどうなるでしょう。

A. `(3, 2, 2)`　B. `(2, 2, 3)`　C. `(1, 3, 2, 2)`

`Dataset` はまだバッチを作らず、transform も呼ばない点に注目してください。

In [ ]:
# STEP 4: 予測の処理を実行し、出力を照合する
prediction_1 = "B"
raw_reference = ReferenceArrayDataset(tiny_images, tiny_labels, transform=None)[0]
print("答え:", prediction_1, "shape:", raw_reference["image"].shape)

### 変える

`to_chw_float` の最後を `[..., ::-1]` のように変更するのではなく、入力のコピーを左右反転してから渡してみてください。元の `tiny_images` が変わらなければ、データ拡張が原本を壊していません。

実データでの典型形は `Image.open(path).convert("RGB")` です。壊れた画像や欠損 path は、事前検証で除くか、明示的な例外に画像 ID を含めます。黙って別画像を返すとラベル対応が壊れます。

### 書く

下の class を完成させてください。`transform` があるときだけ画像へ適用し、辞書を返します。`copy()` は拡張処理から原配列を守るためです。

In [ ]:
# STEP 5: 書くの処理を実行し、出力を照合する
class ArrayVisionDataset:
    def __init__(self, images, labels, transform=None):
        self.images, self.labels, self.transform = images, labels, transform
    def __len__(self):
        # TODO: サンプル数を返す
        return 0
    def __getitem__(self, index):
        # TODO: copy → 任意の transform → image/label の辞書
        return None

learner_dataset = ArrayVisionDataset(tiny_images, tiny_labels, transform=to_chw_float)

In [ ]:
# STEP 6: 書くの処理を実行し、出力を照合する
learner_sample = safe_call(lambda: learner_dataset[0])
check("dataset length", safe_call(lambda: len(learner_dataset)), 2, "labels の長さを返します")
check("CHW shape", safe_call(lambda: safe_get(learner_sample, "image").shape), (3, 2, 2), "transform を呼びます")
check("integer label", safe_get(learner_sample, "label"), 3, "int(labels[index]) を返します")
check("scaled first value", safe_call(lambda: safe_get(learner_sample, "image")[1, 0, 0]), 1 / 255, "CHW の C=1 は元 RGB の G です")

## 2. `DataLoader` は「複数件をまとめる」

`torch.utils.data.DataLoader(dataset, batch_size=..., shuffle=...)` は index を選び、複数の sample をまとめます。標準の画像 tensor は **`(B, C, H, W)`** です。B は batch size。HWC のまま積むと `(B,H,W,C)` になり、多くの PyTorch モデルに渡せません。

`num_workers` は読込プロセス数、`pin_memory=True` は CPU→GPU 転送用メモリを使う指定です。Windows や notebook ではまず `num_workers=0` が安全です。速さは実測してから増やします。

In [ ]:
# STEP 7: 2. `DataLoader` は「複数件をまとめる」の処理を実行し、出力を照合する
def reference_collate(samples):
    images = np.stack([s["image"] for s in samples], axis=0)
    labels = np.asarray([s["label"] for s in samples], dtype=np.int64)
    return {"images": images, "labels": labels}

reference_batch = reference_collate([reference_dataset[0], reference_dataset[1]])
print("batch:", reference_batch["images"].shape, reference_batch["labels"].shape)

def tiny_backbone(batch_bchw):
    # 各チャネルの mean/std を埋め込みにする、極小の特徴抽出器
    means = batch_bchw.mean(axis=(2, 3))
    stds = batch_bchw.std(axis=(2, 3))
    return np.concatenate([means, stds], axis=1)

rng = np.random.default_rng(9)
head_weight = rng.normal(0, 0.1, size=(6, 5))
head_bias = np.zeros(5)
embedding = tiny_backbone(reference_batch["images"])
logits = embedding @ head_weight + head_bias
print("backbone output:", embedding.shape, "head output:", logits.shape)

In [ ]:
# STEP 8: 2. `DataLoader` は「複数件をまとめる」の処理を実行し、出力を照合する
row = train_df.iloc[0]
image_hwc = np.asarray(Image.open(DATA_DIR / "images" / "train" / row["image_id"]).convert("RGB"))
image_chw = to_chw_float(image_hwc)
print("real image HWC:", image_hwc.shape, image_hwc.dtype)
print("model input CHW:", image_chw.shape, image_chw.dtype, f"range=({image_chw.min():.3f}, {image_chw.max():.3f})")

### 予測

shape `(8, 3, 224, 224)` の float32 バッチを backbone に入れ、head が5クラスを返すと logits の shape はどれでしょう。

A. `(8, 5)`　B. `(5, 8)`　C. `(8, 3, 5)`

head は各画像の埋め込み1本をクラス得点へ写します。

In [ ]:
# STEP 9: 予測の処理を実行し、出力を照合する
prediction_2 = "A"
print("答え:", prediction_2, "— batch 次元は保たれ、クラス次元が最後に来ます")

### 変える

`head_weight` を `(6, 2)` にしてみると出力は2クラスになります。これは実 PyTorch の `model.fc = torch.nn.Linear(model.fc.in_features, 2)` に相当します。ResNet は `fc`、MobileNet などは `classifier` と、交換箇所の属性名がモデルごとに違います。

`torchvision.models.resnet18(weights=ResNet18_Weights.DEFAULT)` は学習済み重みを要求し、初回はダウンロードが起こり得ます。本教材では実行しません。`weights=None` はオフラインですが、ランダム初期化なので転移学習ではありません。

### 書く

sample 辞書のリストを、画像とラベルの2配列へまとめる `collate_vision` を完成させてください。

In [ ]:
# STEP 10: 書くの処理を実行し、出力を照合する
def collate_vision(samples):
    # TODO: image を axis=0 に stack、label を int64 配列にする
    return None

learner_batch = safe_call(lambda: collate_vision([reference_dataset[0], reference_dataset[1]]))

In [ ]:
# STEP 11: 書くの処理を実行し、出力を照合する
check("BCHW batch", safe_call(lambda: safe_get(learner_batch, "images").shape), (2, 3, 2, 2), "np.stack(..., axis=0)")
check("label batch", safe_call(lambda: safe_get(learner_batch, "labels").tolist()), [3, 1], "sample 順に label を集めます")
check("label dtype", safe_call(lambda: str(safe_get(learner_batch, "labels").dtype)), "int64", "dtype=np.int64")
check("batch first value", safe_call(lambda: safe_get(learner_batch, "images")[1, 0, 0, 0]), 12 / 255, "2件目の R チャネル先頭です")

## 3. backbone を凍結し、head を学習する

転移モデルは、汎用特徴を作る **backbone** と、今回のクラスへ写す **head** に分けて考えます。

- frozen: backbone を固定し head だけ学習。小データで速く、過学習しにくい
- full fine-tune: 全層を更新。データが十分で、元タスクとの差が大きいと有利になり得る
- staged unfreeze: 最初は head、次に backbone 後段から徐々に解除

PyTorch では `parameter.requires_grad = False` が凍結です。ただし optimizer に `model.parameters()` を無条件で渡さず、`filter(lambda p: p.requires_grad, model.parameters())` と学習対象を明示します。

In [ ]:
# STEP 12: 3. backbone を凍結し、head を学習するの処理を実行し、出力を照合する
parameters = {
    "backbone.kernel": np.zeros((3, 4)),
    "backbone.bias": np.zeros(4),
    "head.weight": np.zeros((4, 5)),
    "head.bias": np.zeros(5),
}
frozen_flags = {
    "backbone.kernel": False,
    "backbone.bias": False,
    "head.weight": True,
    "head.bias": True,
}

def sgd_step(params, grads, trainable, learning_rates):
    updated = {name: value.copy() for name, value in params.items()}
    for name in updated:
        if trainable[name]:
            updated[name] -= learning_rates[name] * grads[name]
    return updated

grads = {name: np.ones_like(value) for name, value in parameters.items()}
lrs = {name: (1e-3 if name.startswith("backbone") else 1e-2) for name in parameters}
after = sgd_step(parameters, grads, frozen_flags, lrs)
print("backbone changed:", bool(np.any(after["backbone.kernel"] != 0)))
print("head changed:", bool(np.any(after["head.weight"] != 0)))

In [ ]:
# STEP 13: 3. backbone を凍結し、head を学習するの処理を実行し、出力を照合する
all_count = sum(value.size for value in parameters.values())
head_count = sum(parameters[name].size for name, flag in frozen_flags.items() if flag)
print(f"all={all_count:,}, trainable_when_frozen={head_count:,}")
print("optimizer へ渡す名前:", [name for name, flag in frozen_flags.items() if flag])

### 予測

backbone の学習率を head の10分の1にして全層を更新する設定は何を狙っているでしょう。

A. 汎用特徴を小さく調整し、head を大きく適応させる

B. backbone を完全に凍結する

C. GPU メモリを必ず半分にする

In [ ]:
# STEP 14: 予測の処理を実行し、出力を照合する
prediction_3 = "A"
print("答え:", prediction_3, "— 層ごとに学習率を変える discriminative learning rate の考え方です")

### 変える

`frozen_flags` の backbone 2項目を `True` にすると full fine-tune です。どの値が変わるかを `sgd_step` で確認してください。

注意: `model.eval()` は dropout / batch normalization を推論モードにする操作で、凍結とは別です。`requires_grad=False` でも、モデル全体を `train()` にすると batch normalization の移動統計は変わり得ます。

### 書く

パラメータ辞書と `requires_grad` 相当の flags から、学習対象の名前と要素数を返してください。C# なら `Where(...).Sum(x => x.Length)` に相当します。

In [ ]:
# STEP 15: 書くの処理を実行し、出力を照合する
def select_trainable(params, flags):
    # TODO: flags[name] が True の項目だけ選び、names と count を返す
    return None

learner_selection = safe_call(lambda: select_trainable(parameters, frozen_flags))
full_selection = safe_call(lambda: select_trainable(parameters, {name: True for name in parameters}))
empty_selection = safe_call(lambda: select_trainable(parameters, {name: False for name in parameters}))

In [ ]:
# STEP 16: 書くの処理を実行し、出力を照合する
check("frozen names", safe_get(learner_selection, "names"), ["head.weight", "head.bias"], "True の名前だけ残します")
check("frozen count", safe_get(learner_selection, "count"), 25, "20 + 5 要素です")
check("full count", safe_get(full_selection, "count"), 41, "全4配列の size を足します")
check("empty count", safe_get(empty_selection, "count"), 0, "学習対象なしなら0です")

## 4. TTA は「同じ画像の予測を平均」する

Test-Time Augmentation (TTA) は、原画像・左右反転など複数 view を推論し、**確率を平均**します。学習は増えませんが、推論回数は view 数倍です。

訓練 transform はランダムでも、TTA の view は再現可能な固定集合にします。このデータの回転を使うなら生成条件と同じ **±25度以内**に留めます。ラベルの意味が変わる拡張は使いません。

In [ ]:
# STEP 17: 4. TTA は「同じ画像の予測を平均」するの処理を実行し、出力を照合する
# shape: (views, batch, classes)。各行は softmax 後の確率です。
tta_predictions = np.array([
    [[0.60, 0.30, 0.10], [0.20, 0.50, 0.30]],
    [[0.45, 0.45, 0.10], [0.10, 0.55, 0.35]],
    [[0.51, 0.34, 0.15], [0.15, 0.45, 0.40]],
], dtype=np.float64)

reference_tta = tta_predictions.mean(axis=0)
print("views:", tta_predictions.shape)
print("averaged:", reference_tta.round(3))
print("classes:", reference_tta.argmax(axis=1))

In [ ]:
# STEP 18: 4. TTA は「同じ画像の予測を平均」するの処理を実行し、出力を照合する
def input_megabytes(batch, channels, height, width, bytes_per_value=4):
    return batch * channels * height * width * bytes_per_value / (1024 ** 2)

mb_224 = input_megabytes(32, 3, 224, 224)
mb_112 = input_megabytes(32, 3, 112, 112)
print(f"input only: 32x3x224x224 float32 = {mb_224:.2f} MiB")
print(f"half width/height:              = {mb_112:.2f} MiB ({mb_112/mb_224:.2f}x)")
print("3-view TTA inference calls: 3x")

### 予測

画像の縦横を 224 から 112 に半減すると、入力配列の要素数は何倍になりますか。

A. 1/2　B. 1/4　C. 1/8

チャネル数と batch size は同じとします。

In [ ]:
# STEP 19: 予測の処理を実行し、出力を照合する
prediction_4 = "B"
print("答え:", prediction_4, "— 面積なので (112/224)^2 = 1/4 です")

### 変える

TTA を1 view、2 views、3 viewsと増やし、予測 class がいつ安定するかを確認してください。平均で改善しない view はコストだけ増やします。

GPU は行列計算を並列化しますが、model・勾配・optimizer state・中間 activation が GPU メモリを使います。OOM なら最初に batch / 解像度を下げ、mixed precision や gradient accumulation を検討します。CPU では小バッチ・frozen embedding の事前計算が現実的です。

### 書く

`(views, batch, classes)` の確率を view 軸で平均する `average_tta` を完成させてください。logits の平均と確率の平均は一般に同じではないため、入力がどちらかを統一します。

In [ ]:
# STEP 20: 書くの処理を実行し、出力を照合する
def average_tta(probabilities):
    # TODO: view 軸(axis=0)で平均する
    return None

learner_tta = safe_call(lambda: average_tta(tta_predictions))

In [ ]:
# STEP 21: 書くの処理を実行し、出力を照合する
check("TTA shape", safe_call(lambda: learner_tta.shape), (2, 3), "view 軸だけが消えます")
check("TTA first row", safe_call(lambda: learner_tta[0]), [0.52, 0.3633333333, 0.1166666667], "3 view の列ごとの平均です")
check("TTA classes", safe_call(lambda: learner_tta.argmax(axis=1).tolist()), [0, 1], "最後に class 軸 argmax")
check("probability sums", safe_call(lambda: learner_tta.sum(axis=1)), [1.0, 1.0], "確率平均なら行和は1です")

## 5. accuracy の次に confusion matrix を見る

`sklearn.metrics.confusion_matrix(y_true, y_pred)` は、行を正解、列を予測として件数を集計します。accuracy だけでは「どのクラスを何と間違えたか」が見えません。

- precision: そのクラスと予測した中で正解した割合
- recall: そのクラスの正解例を拾えた割合
- macro: 各クラスを同じ重みで平均
- weighted: 各クラスの件数で重み付け
- micro: 全件の TP/FP/FN をまとめて計算（単一ラベル多クラスでは accuracy と一致しやすい）

分割は unit08 と同じく product 単位です。同一商品の近似画像を train / validation にまたがせてはいけません。

In [ ]:
# STEP 22: 5. accuracy の次に confusion matrix を見るの処理を実行し、出力を照合する
class_names = sorted(train_df["label"].unique().tolist())
label_to_id = {label: index for index, label in enumerate(class_names)}
preview = train_df.head(15).copy()
y_true_real = preview["label"].map(label_to_id).to_numpy()
y_pred_real = y_true_real.copy()
y_pred_real[[1, 4, 8]] = (y_pred_real[[1, 4, 8]] + 1) % len(class_names)

cm_real = confusion_matrix(y_true_real, y_pred_real, labels=np.arange(len(class_names)))
precision, recall, f1, support = precision_recall_fscore_support(
    y_true_real, y_pred_real, labels=np.arange(len(class_names)), zero_division=0
)
print("confusion matrix (row=true, col=pred):")
print(cm_real)
print("macro F1:", round(float(f1.mean()), 3))

errors = preview.loc[y_true_real != y_pred_real, ["image_id", "product_key", "label"]].copy()
errors["predicted"] = [class_names[i] for i in y_pred_real[y_true_real != y_pred_real]]
errors

In [ ]:
# STEP 23: 5. accuracy の次に confusion matrix を見るの処理を実行し、出力を照合する
verified_scores = pd.DataFrame([
    ("color histogram + LinearSVC", 0.3533),
    ("HOG + LinearSVC", 0.7840),
    ("scratch CNN: poor recipe", 0.5731),
    ("scratch CNN: BN + augmentation + cosine", 0.9038),
    ("transfer: full fine-tune", 0.9451),
    ("transfer: frozen backbone + linear head", 0.9538),
], columns=["recipe", "grouped_cv_accuracy"])
verified_scores

### 予測

この検証表だけから「scratch CNN の限界は 0.5731」と結論できるでしょうか。

A. できる　B. できない

同じ scratch でも学習 recipe が違う行を比較してください。

In [ ]:
# STEP 24: 予測の処理を実行し、出力を照合する
prediction_5 = "B"
print("答え:", prediction_5)
print("0.5731 は不十分な recipe、改善版 scratch は 0.9038。転移は 0.9451 / 0.9538 です。")

### 変える

`errors` を true / predicted の組で並べ、画像を数件ずつ見てください。背景色、角度、商品 group、画像破損など共通原因を仮説にします。次の改善は accuracy の小数だけでなく、この誤りのまとまりから決めます。

この小データでは frozen backbone + linear head の 0.9538 が full fine-tune の 0.9451 を上回りました。これは普遍則ではなく、「データ量・元タスクとの近さ・正則化・計算予算」に応じて freeze 方針を比較する根拠です。

### 書く

正解 ID と予測 ID から confusion matrix を作ってください。`matrix[true, pred]` を1ずつ増やします。`np.add.at` は重複 index も漏らさず加算する NumPy API です。

In [ ]:
# STEP 25: 書くの処理を実行し、出力を照合する
def confusion_counts(y_true, y_pred, n_classes):
    # TODO: (n_classes, n_classes) の int 配列を作り、true/pred 位置を加算
    return None

toy_true = np.array([0, 0, 1, 1, 2, 2, 2])
toy_pred = np.array([0, 1, 1, 1, 2, 0, 2])
learner_cm = safe_call(lambda: confusion_counts(toy_true, toy_pred, 3))

In [ ]:
# STEP 26: 書くの処理を実行し、出力を照合する
expected_cm = np.array([[1, 1, 0], [0, 2, 0], [1, 0, 2]])
check("confusion shape", safe_call(lambda: learner_cm.shape), (3, 3), "クラス数×クラス数です")
check("confusion values", learner_cm, expected_cm, "matrix[true, pred] を加算します")
check("accuracy from diagonal", safe_call(lambda: np.trace(learner_cm) / learner_cm.sum()), 5 / 7, "対角和 / 全件数")
check("class-0 recall", safe_call(lambda: learner_cm[0, 0] / learner_cm[0].sum()), 0.5, "正解 class 0 の行で割ります")

<!-- REAL_MODEL_SECTION_UNIT09 -->
## 実ライブラリで確認: ResNet18 の head を交換して固定する

`torchvision.models.resnet18` は画像特徴を抽出する畳み込みネットワークです。転移学習では、既存の `fc`（最後の分類 head）を新しいクラス数へ交換し、最初は backbone を固定します。これは C# で大きなサービスの実装を再利用し、末端の Strategy だけ差し替える感覚に近いです。

本番では `ResNet18_Weights.DEFAULT` で事前学習済み重みを使えます。この教材の検証はダウンロード不要にするため `weights=None` で構造と API を確認します。

In [ ]:
# STEP 1: 実DatasetとDataLoaderでBCHW batchを作る
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18

torch.set_num_threads(1)
class TinyTorchVisionDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {"image": self.images[index], "label": self.labels[index]}

torch_images = torch.randn(4, 3, 32, 32)
torch_labels = torch.tensor([0, 1, 0, 1])
torch_loader = DataLoader(TinyTorchVisionDataset(torch_images, torch_labels), batch_size=2, shuffle=False)
torch_batch = next(iter(torch_loader))
print("BCHW:", tuple(torch_batch["image"].shape))

# STEP 2: ResNet18のheadを交換し、fcだけ学習可能にする
real_resnet = resnet18(weights=None)
real_resnet.fc = nn.Linear(real_resnet.fc.in_features, 2)
for real_name, real_parameter in real_resnet.named_parameters():
    real_parameter.requires_grad = real_name.startswith("fc.")
real_optimizer = torch.optim.SGD(
    [parameter for parameter in real_resnet.parameters() if parameter.requires_grad],
    lr=0.05,
)
print("frozen trainable:", [name for name, p in real_resnet.named_parameters() if p.requires_grad])

# STEP 3: CrossEntropyLoss→backward→optimizer.stepを1回通す
real_resnet.train()
real_optimizer.zero_grad()
real_logits = real_resnet(torch_batch["image"])
real_loss = nn.CrossEntropyLoss()(real_logits, torch_batch["label"])
real_loss.backward()
real_optimizer.step()
print("logits / loss:", tuple(real_logits.shape), round(float(real_loss.detach()), 4))

# STEP 4: 次段階ではlayer4とfcを低い学習率で解凍する
for real_name, real_parameter in real_resnet.named_parameters():
    real_parameter.requires_grad = real_name.startswith(("layer4.", "fc."))
real_staged_names = [name for name, p in real_resnet.named_parameters() if p.requires_grad]
print("staged parameter tensors:", len(real_staged_names))
print("first/last:", real_staged_names[0], real_staged_names[-1])

## 振り返り

次を自分の言葉で1〜2文ずつ説明してください。

1. `Dataset` と `DataLoader` の責務を分ける利点は何ですか。
2. frozen backbone が小データで full fine-tune より良いことがあるのはなぜですか。
3. TTA の精度上の利点と、推論コスト上の欠点は何ですか。
4. accuracy が同じ2モデルでも confusion matrix を見る必要があるのはなぜですか。

C# の設計に置き換えるなら、差し替え可能な transform、backbone/head の責務分離、更新対象の明示は、依存性注入・単一責任・安全な state 管理の延長です。

## まとめと次の一歩

- 画像1件は `Dataset`、BCHW バッチ化は `DataLoader`
- backbone は汎用埋め込み、head は今回のクラスへの写像
- freeze は `requires_grad` と optimizer 対象の両方を確認
- full / frozen / staged はデータ量と計算予算で比較
- TTA は確率を view 間平均し、推論回数との交換条件を測る
- confusion matrix と誤分類画像から次の改善を決める

すべてのチェックが `OK` になったら、演習ではファイル path を扱う Dataset、head 差し替え、学習・評価、最後に小さな画像分類パイプラインへ進みます。